# 📊 Exploratory Data Analysis — Student Placement Prediction

This notebook explores the `data/placement.csv` dataset: distributions, relationships between features and placement outcome, and correlations. Insights from this notebook informed feature selection for `src/train.py`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
%matplotlib inline

df = pd.read_csv("../data/placement.csv")
df.head()

## 1. Basic Info & Data Quality

In [ ]:
print("Shape:", df.shape)
df.info()

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

In [ ]:
df_clean = df.drop_duplicates().copy()
for col in df_clean.select_dtypes(include=np.number).columns:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())
df_clean.dropna(subset=["Placement"], inplace=True)
df_clean.describe()

## 2. Placement Distribution

In [ ]:
plt.figure(figsize=(5,4))
sns.countplot(data=df_clean, x="Placement", palette="Set2")
plt.title("Placement Distribution")
plt.show()

print(df_clean["Placement"].value_counts(normalize=True).round(3) * 100)

## 3. CGPA vs Placement

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(data=df_clean, x="Placement", y="CGPA", palette="Set3")
plt.title("CGPA vs Placement")
plt.show()

## 4. Internships vs Placement

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df_clean, x="Internships", hue="Placement", palette="coolwarm")
plt.title("Internships vs Placement")
plt.show()

## 5. Communication Skills vs Placement

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(data=df_clean, x="Placement", y="Communication_Skills", palette="Pastel1")
plt.title("Communication Skills vs Placement")
plt.show()

## 6. Correlation Heatmap

In [ ]:
plt.figure(figsize=(10,8))
numeric_df = df_clean.copy()
numeric_df["Placement_Num"] = numeric_df["Placement"].map({"Yes": 1, "No": 0})
corr = numeric_df.select_dtypes(include=np.number).corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Correlation Heatmap")
plt.show()

## 7. Distributions (Histograms)

In [ ]:
num_cols = ["CGPA", "IQ", "Previous_Semester_Percentage", "Aptitude_Score",
            "Technical_Skills_Score", "Attendance_Percentage"]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.flatten(), num_cols):
    sns.histplot(df_clean[col], kde=True, ax=ax, color="steelblue")
    ax.set_title(f"Distribution of {col}")
plt.tight_layout()
plt.show()

## 8. Boxplots for Outlier Detection

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.flatten(), num_cols):
    sns.boxplot(y=df_clean[col], ax=ax, color="lightcoral")
    ax.set_title(f"Boxplot of {col}")
plt.tight_layout()
plt.show()

## 9. Key Takeaways

- Students with higher **CGPA**, more **internships**, and stronger **communication skills** show a noticeably higher placement rate.
- **Backlogs** show a negative relationship with placement.
- The correlation heatmap confirms CGPA, Technical Skills Score, and Internships are among the strongest predictors of placement.
- These insights guided feature selection in `src/preprocess.py` and `src/train.py`.